In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Keep this in sync with --history_length used when generating/evaluating the rollouts.
history_length = 3
workspace = Path.cwd()
if not (workspace / "outputs" / "wm-rollouts").exists():
    workspace = workspace.parent

# Edit these paths and labels to compare different rollouts.
ollout_summaries = {
    # "pretrained-original": workspace / "outputs/wm-rollouts/pusht-pretrained-original/metrics_summary.csv",
    # "pretrained-small": workspace / "outputs/wm-rollouts/pusht-pretrained-small/metrics_summary.csv",
    # "finetuned-original": workspace / "outputs/wm-rollouts/pusht-finetuned-original/20261023_101236-NanoWM-B-2-pusht-ER_False-S20000-LR1e-05_cosine/metrics_summary.csv",
    # "finetuned-small": workspace / "outputs/wm-rollouts/pusht-finetuned-small/20261023_101236-NanoWM-B-2-pusht-ER_False-S20000-LR1e-05_cosine/metrics_summary.csv",
    "finetuned-ER-original": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-original/20261023_124443-NanoWM-B-2-pusht-ER_True-S40000-LR1e-05_cosine/metrics_summary.csv",
    "finetuned-ER-small": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-small/20261023_124443-NanoWM-B-2-pusht-ER_True-S40000-LR1e-05_cosine/metrics_summary.csv",
    # "finetuned-ER-original-R0.1": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-original/20261026_174733-NanoWM-B-2-pusht-ER_True-S22000-R0.1-B1.0/metrics_summary.csv",
    # "finetuned-ER-small-R0.1": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-small/20261026_174733-NanoWM-B-2-pusht-ER_True-S22000-R0.1-B1.0/metrics_summary.csv",
    # "finetuned-ER-original-B0.1": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-original/20261026_175527-NanoWM-B-2-pusht-ER_True-S40000-R0.5-B0.1/metrics_summary.csv",
    # "finetuned-ER-small-B0.1": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-small/20261026_175527-NanoWM-B-2-pusht-ER_True-S40000-R0.5-B0.1/metrics_summary.csv",
    "finetuned-ER-original-R0.1-B0.1-": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-original/20261027_112004-NanoWM-B-2-pusht-ER_True-S22000-R0.1-B0.1/metrics_summary.csv",
    "finetuned-ER-small-R0.1-B0.1": workspace / "outputs/wm-rollouts/pusht-finetuned-ER-small/20261027_112004-NanoWM-B-2-pusht-ER_True-S22000-R0.1-B0.1/metrics_summary.csv",
}
metric_names = ["mse", "psnr", "ssim", "lpips", "dreamsim"]

# Load each summary and remove context frames before plotting or computing means.
rollout_data = {}
for label, summary_path in rollout_summaries.items():
    if not summary_path.exists():
        raise FileNotFoundError(f"Metrics summary not found for {label}: {summary_path}")

    summary = pd.read_csv(summary_path)
    missing_metrics = set(["frame_idx", *metric_names]) - set(summary.columns)
    if missing_metrics:
        raise ValueError(f"Missing columns in {summary_path}: {sorted(missing_metrics)}")

    generated = summary.loc[summary["frame_idx"] >= history_length].sort_values("frame_idx")
    if generated.empty:
        raise ValueError(f"No generated frames found in {summary_path}")
    rollout_data[label] = generated

fig, axes = plt.subplots(
    len(metric_names), 1, figsize=(11, 13), sharex=True, constrained_layout=True
)
fig.suptitle("World model rollout metric comparison (context frames excluded)", fontsize=14)

for ax, metric in zip(axes, metric_names):
    for label, generated in rollout_data.items():
        # The first generated frame is displayed as rollout step 1.
        rollout_steps = generated["frame_idx"] - history_length + 1
        values = generated[metric]
        line, = ax.plot(rollout_steps, values, marker="o", linewidth=1.8, label=label)
        mean_value = values.mean()
        ax.axhline(
            mean_value,
            color=line.get_color(),
            linestyle="--",
            linewidth=1.5,
            label=f"{label} mean: {mean_value:.4g}",
        )

    ax.set_ylabel(metric.upper())
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

axes[-1].set_xlabel("Generated rollout frame")
plt.show()


In [ ]:
# Set the desired output location here.
plot_path = workspace / "outputs/wm-rollouts/metric-comparison/pretrained-original-vs-finetuned-ER.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"Saved plot to: {plot_path.resolve()}")
